# 🔥❄️ Snow Fire Fight — Hand Gesture Game

A real-time hand gesture game using your webcam.

| Gesture | Attack | How To |
|---|---|---|
| ✊ Fist | ❄️ Snow | Close all 4 fingers |
| 👉 Swipe | 🔥 Fire | Flick index finger left → right |

**Goal:** Shoot the opposite element to cancel enemy projectiles and score points.
- Fire cancels Snow ✅
- Snow cancels Fire ✅
- Same type = no cancel ❌

> **Requirements:** Webcam, `assets/fire.wav` and `assets/snow.wav` in the same folder.

## ⚙️ 1. Install Dependencies

In [ ]:
!pip install opencv-python mediapipe pygame --quiet
print("✅ All dependencies installed")

## 📦 2. Imports & Constants

In [ ]:
import math
import cv2
import mediapipe as mp
import pygame
import time
import random

# ── Screen & projectile settings ──────────────────────────────────────────────
SCREEN_WIDTH      = 640
SCREEN_HEIGHT     = 480
PROJECTILE_RADIUS = 10
PROJECTILE_SPEED  = 5
ENEMY_SPAWN_RATE  = 2   # seconds between enemy projectiles
ATTACK_COOLDOWN   = 1   # seconds between player attacks
SWIPE_THRESHOLD   = 20  # pixels of horizontal movement to count as a swipe

# ── Colors (BGR) ──────────────────────────────────────────────────────────────
COLOR_FIRE  = (0, 0, 255)      # red
COLOR_SNOW  = (255, 200, 200)  # light blue
COLOR_TEXT  = (0, 0, 0)
COLOR_LABEL = (50, 200, 50)

print("✅ Constants set")

## 🔊 3. Audio & MediaPipe Setup

In [ ]:
pygame.mixer.init()

try:
    fire_sound = pygame.mixer.Sound("assets/fire.wav")
    snow_sound = pygame.mixer.Sound("assets/snow.wav")
    print("✅ Sound files loaded")
except FileNotFoundError:
    print("⚠️  Sound files not found in assets/ — game will run silently.")
    fire_sound = None
    snow_sound = None

mp_hands = mp.solutions.hands
hands    = mp_hands.Hands(max_num_hands=1)
mp_draw  = mp.solutions.drawing_utils

print("✅ MediaPipe hands model ready")

## 🤚 4. Gesture Detection Functions

In [ ]:
def is_fist(lm) -> bool:
    """
    Returns True if all 4 fingers are folded (✊ fist = SNOW attack).
    Checks if fingertip y > middle-knuckle y (folded downward).
    """
    tips = [8, 12, 16, 20]  # index, middle, ring, pinky tips
    folded = sum(1 for tip in tips if lm[tip].y > lm[tip - 2].y)
    return folded == 4


def is_swipe(current_x: int, prev_x: int, index_up: bool) -> bool:
    """
    Returns True if the index finger is up and moved right by SWIPE_THRESHOLD
    pixels (👉 rightward swipe = FIRE attack).
    """
    return index_up and (current_x - prev_x > SWIPE_THRESHOLD)


def detect_gesture(lm, prev_x: int):
    """
    Evaluate landmarks and return (gesture_str | None, hand_x, hand_y).
    Priority: fist (snow) > swipe (fire).
    """
    index_up = lm[8].y < lm[6].y
    x = int(lm[8].x * SCREEN_WIDTH)
    y = int(lm[8].y * SCREEN_HEIGHT)

    if is_fist(lm):
        return "snow", x, y
    elif is_swipe(x, prev_x, index_up):
        return "fire", x, y
    return None, x, y

print("✅ Gesture functions defined")

## 🎮 5. Game Logic Functions

In [ ]:
def draw_projectile(img, x: float, y: float, color: tuple) -> None:
    """Draw a filled circle on the frame at (x, y)."""
    cv2.circle(img, (int(x), int(y)), PROJECTILE_RADIUS, color, -1)


def spawn_enemy_projectile() -> dict:
    """Create a new enemy projectile entering from the right side."""
    return {
        'x': SCREEN_WIDTH + PROJECTILE_RADIUS,
        'y': random.randint(100, SCREEN_HEIGHT - 100),
        'type': random.choice(['fire', 'snow'])
    }


def check_collision(proj1: dict, proj2: dict) -> bool:
    """Euclidean distance collision check between two projectile dicts."""
    dist = math.sqrt((proj1['x'] - proj2['x'])**2 + (proj1['y'] - proj2['y'])**2)
    return dist < PROJECTILE_RADIUS * 2


def play_sound(gesture: str) -> None:
    """Play the sound for the given gesture if sound files are loaded."""
    if gesture == "fire" and fire_sound:
        fire_sound.play()
    elif gesture == "snow" and snow_sound:
        snow_sound.play()


def draw_hud(img, score: int, gesture: str | None) -> None:
    """Overlay score and current gesture label onto the frame."""
    cv2.putText(img, f"Score: {score}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, COLOR_TEXT, 2)
    if gesture:
        label = "✊ SNOW" if gesture == "snow" else "👉 FIRE"
        cv2.putText(img, label, (10, 65),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, COLOR_LABEL, 2)
    cv2.putText(img, "Press X to quit", (SCREEN_WIDTH - 170, SCREEN_HEIGHT - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 100, 100), 1)

print("✅ Game logic functions defined")

## 🕹️ 6. Main Game Loop

> **Run this cell to start the game.**  
> A separate OpenCV window will open.  
> **Press `X`** in that window to quit.

In [ ]:
# ── Game state ─────────────────────────────────────────────────────────────
prev_x            = 0
last_trigger_time = 0
last_enemy_spawn  = time.time()
score             = 0
player_projectiles = []
enemy_projectiles  = []

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("❌ Could not open webcam. Check your camera index.")

print("🎮 Game started! OpenCV window should appear.")
print("   ✊ Fist  → ❄️  Snow attack")
print("   👉 Swipe → 🔥 Fire attack")
print("   Press X in the game window to quit.")

# ── Main loop ──────────────────────────────────────────────────────────────
while True:
    success, img = cap.read()
    if not success:
        print("⚠️  Frame read failed, retrying...")
        continue

    img = cv2.flip(img, 1)
    img = cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result  = hands.process(img_rgb)

    # ── Spawn enemy projectile every ENEMY_SPAWN_RATE seconds ──────────────
    if time.time() - last_enemy_spawn > ENEMY_SPAWN_RATE:
        enemy_projectiles.append(spawn_enemy_projectile())
        last_enemy_spawn = time.time()

    gesture = None

    # ── Hand detection & gesture recognition ───────────────────────────────
    if result.multi_hand_landmarks:
        for handLms in result.multi_hand_landmarks:
            lm = handLms.landmark
            mp_draw.draw_landmarks(img, handLms, mp_hands.HAND_CONNECTIONS)

            gesture, x, y = detect_gesture(lm, prev_x)
            prev_x = x

            # Fire attack on cooldown
            if gesture and time.time() - last_trigger_time > ATTACK_COOLDOWN:
                play_sound(gesture)
                print(f"  [{time.strftime('%H:%M:%S')}] Gesture: {gesture.upper()} at ({x}, {y})")
                player_projectiles.append({'x': x, 'y': y, 'type': gesture})
                last_trigger_time = time.time()

    # ── Move & draw player projectiles (travel right → ) ───────────────────
    for p in player_projectiles[:]:
        p['x'] += PROJECTILE_SPEED
        color = COLOR_FIRE if p['type'] == 'fire' else COLOR_SNOW
        draw_projectile(img, p['x'], p['y'], color)
        if p['x'] > SCREEN_WIDTH + PROJECTILE_RADIUS:
            player_projectiles.remove(p)

    # ── Move & draw enemy projectiles (travel left ← ) ─────────────────────
    for e in enemy_projectiles[:]:
        e['x'] -= PROJECTILE_SPEED
        color = COLOR_FIRE if e['type'] == 'fire' else COLOR_SNOW
        draw_projectile(img, e['x'], e['y'], color)
        if e['x'] < -PROJECTILE_RADIUS:
            enemy_projectiles.remove(e)

    # ── Collision detection — opposite types cancel & score ─────────────────
    for p in player_projectiles[:]:
        for e in enemy_projectiles[:]:
            if check_collision(p, e):
                opposite = (p['type'] == 'fire' and e['type'] == 'snow') or \
                           (p['type'] == 'snow' and e['type'] == 'fire')
                if opposite:
                    score += 10
                    print(f"  💥 Hit! Score: {score}")
                    if p in player_projectiles: player_projectiles.remove(p)
                    if e in enemy_projectiles:  enemy_projectiles.remove(e)

    # ── HUD overlay ─────────────────────────────────────────────────────────
    draw_hud(img, score, gesture)

    cv2.imshow("🔥❄️ SNOW FIRE FIGHT", img)
    if cv2.waitKey(1) & 0xFF == ord('x'):
        break

cap.release()
cv2.destroyAllWindows()
print(f"\n🏁 Game over! Final score: {score}")

## 🔧 7. Configuration — Tweak the Game

Change these values and re-run **Cell 2** (Imports & Constants) to apply.

In [ ]:
# ── Difficulty presets ────────────────────────────────────────────────────────

PRESET = "medium"  # Change to: "easy" | "medium" | "hard"

presets = {
    "easy":   dict(PROJECTILE_SPEED=3, ENEMY_SPAWN_RATE=3, ATTACK_COOLDOWN=0.5),
    "medium": dict(PROJECTILE_SPEED=5, ENEMY_SPAWN_RATE=2, ATTACK_COOLDOWN=1.0),
    "hard":   dict(PROJECTILE_SPEED=8, ENEMY_SPAWN_RATE=1, ATTACK_COOLDOWN=1.5),
}

cfg = presets[PRESET]
PROJECTILE_SPEED  = cfg["PROJECTILE_SPEED"]
ENEMY_SPAWN_RATE  = cfg["ENEMY_SPAWN_RATE"]
ATTACK_COOLDOWN   = cfg["ATTACK_COOLDOWN"]

print(f"✅ Preset '{PRESET}' applied:")
print(f"   Projectile speed : {PROJECTILE_SPEED} px/frame")
print(f"   Enemy spawn rate : every {ENEMY_SPAWN_RATE}s")
print(f"   Attack cooldown  : {ATTACK_COOLDOWN}s")